# 02 — WordCount with Kubernetes Executors (Client Mode)

**What runs where:**
- **Driver:** this Notebook's Jupyter kernel, inside your Pod `jupyter-<user>` in namespace `user-<user>`.
- **Executors:** separate Pods in the **same namespace**, created by the Driver through the Kubernetes API with your ServiceAccount `spark-runner`. It may only act inside `user-<user>`.
- **Network path back:** Executors connect to the Driver at this Pod's IP (`SPARK_LAB_DRIVER_HOST`) on random ports, so several kernels in this Pod can each run a Client-Mode session.

**Observe (on the Mac, in a second terminal):**
```bash
kubectl -n user-<user> get pods -l spark-role=executor -w
```
Spark UI → Executors tab: `http://localhost:8080/spark-ui/<user>/executors/` (only the session holding port 4040 is forwarded).

Resource quota: your namespace may request at most 4 CPU / 8Gi. Each Executor here requests 0.5 CPU / ~1.4Gi; extra Executors beyond the quota are not created.

> This folder is refreshed from the runtime image each time the Notebook Server starts. Copy the notebook elsewhere under `work/` before editing it.

In [ ]:
import os, socket
from pyspark.sql import SparkSession

RUNTIME_IMAGE = os.environ.get("SPARK_LAB_RUNTIME_IMAGE", "spark-lab-runtime:0.3.0")

spark = (
    SparkSession.builder
    .appName("WordCount-K8s-Client")
    .master("k8s://https://kubernetes.default.svc:443")
    .config("spark.submit.deployMode", "client")
    # Your own namespace; the ServiceAccount cannot create Pods anywhere else.
    .config("spark.kubernetes.namespace", os.environ["SPARK_LAB_NAMESPACE"])
    # Driver network identity: this Pod's IP, random driver/blockManager ports
    .config("spark.driver.host", os.environ["SPARK_LAB_DRIVER_HOST"])
    .config("spark.driver.bindAddress", "0.0.0.0")
    # Owner reference: Executor Pods are garbage-collected together with this Pod
    .config("spark.kubernetes.driver.pod.name", socket.gethostname())
    # Executors use the same runtime image as this Notebook
    .config("spark.kubernetes.container.image", RUNTIME_IMAGE)
    .config("spark.kubernetes.container.image.pullPolicy", "Never")
    .config("spark.executor.instances", "2")
    .config("spark.executor.memory", "1g")
    .config("spark.executor.cores", "1")
    .config("spark.kubernetes.executor.request.cores", "500m")
    .getOrCreate()
)
sc = spark.sparkContext
print("master     :", sc.master)
print("deployMode :", sc.getConf().get("spark.submit.deployMode"))
print("namespace  :", sc.getConf().get("spark.kubernetes.namespace"))
print("driver Pod :", socket.gethostname(), " ip:", os.environ["SPARK_LAB_DRIVER_HOST"], " kernel pid:", os.getpid())

# sc.uiWebUrl (e.g. http://jupyter-alice:4040) is the in-cluster address; the
# browser reaches the same UI through the lab portal at /spark-ui/<user>/.
user = os.environ.get("SPARK_USER", "<user>")
ui_port = sc.uiWebUrl.rsplit(":", 1)[-1]
if ui_port == "4040":
    print(f"Spark UI     : http://localhost:8080/spark-ui/{user}/  (or http://<mac-ip>:8080/spark-ui/{user}/)")
else:
    print(f"Spark UI     : port {ui_port} is NOT forwarded; another SparkSession in this Pod already holds 4040."
          f" Stop it (or restart that kernel) to see this application at /spark-ui/{user}/.")

Executor Pods start asynchronously. Run the next cell; it waits until both Executors have registered with the Driver.

In [ ]:
import time
for _ in range(60):
    registered = sc._jsc.sc().getExecutorMemoryStatus().size() - 1   # minus the Driver
    if registered >= 2:
        break
    time.sleep(2)
print("registered executors:", registered)

In [ ]:
lines = sc.parallelize(
    [
        "spark on kubernetes client mode",
        "interactive spark notebook with dynamic executors",
        "driver in notebook pod executors in their own pods",
        "spark executors report back to the driver",
    ],
    4,
)
counts = (
    lines.flatMap(lambda line: line.split())
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
)
print(sorted(counts.collect()))

## Map Spark Executors to Kubernetes Pods

Tasks report the host name of the process that ran them. In a Pod the host name is the Pod name, so the result lists the Executor Pods (not `jupyter-<user>`). The Spark REST API shows the same Executors with their Pod IPs. Compare with `kubectl -n user-<user> get pods -l spark-role=executor -o wide`.

In [ ]:
hosts = sc.parallelize(range(8), 8).map(lambda _: socket.gethostname()).distinct().collect()
print("tasks ran on:", sorted(hosts))

import json, urllib.request
base = sc.uiWebUrl.rstrip("/")
app_id = sc.applicationId
with urllib.request.urlopen(f"{base}/api/v1/applications/{app_id}/executors") as r:
    for e in json.load(r):
        print(f"executor id={e['id']:>6}  hostPort={e['hostPort']:<28} completedTasks={e['totalTasks']}")

## Python dependencies are not shared automatically

The next cell compares the Python interpreter on the Driver with the one on an Executor. They match only because both Pods use the same runtime image. A `pip install` in this Notebook changes the Driver only.

In [ ]:
import sys
driver_py = sys.version.split()[0]
executor_py = sc.parallelize([0], 1).map(lambda _: __import__("sys").version.split()[0]).collect()[0]
print("driver python  :", driver_py)
print("executor python:", executor_py)

In [ ]:
# Stopping the SparkSession makes the Driver delete its Executor Pods.
spark.stop()